In [1]:
import torch
from torch.utils.data import DataLoader 
import torchvision
import torch.nn.functional as F
from torchvision import transforms, datasets
from tqdm import tqdm
from autoattack import AutoAttack
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else"cpu")
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [3]:
torch.manual_seed(0)
if device == 'cuda':
    torch.cuda.manual_seed(0)

In [4]:
transforms_ = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
])
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],     # ImageNet mean
    std=[0.229, 0.224, 0.225]       # ImageNet std
)

In [5]:
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device).eval()

Using cache found in C:\Users\theho/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\theho/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\theho/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\theho/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


KeyboardInterrupt: 

In [ ]:
train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_)
test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_)

In [ ]:
train_loader = DataLoader(train_data, batch_size=256, shuffle=False, num_workers=4)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=4)

In [ ]:
prototype_sums   = torch.zeros(10, 1024, device=device)
prototype_counts = torch.zeros(10, device=device)

PROTOTYPE_PATH = "./dinov2_prototypes.pt"

if os.path.exists(PROTOTYPE_PATH):
    prototypes = torch.load(PROTOTYPE_PATH, map_location=device)
    print(f'Prototypes loaded from {PROTOTYPE_PATH}')
else:
    with torch.no_grad():
        for images, labels in tqdm(train_loader):
            images = images.to(device)
            labels = labels.to(device)
    
            images_norm = normalize(images)
            embeddings  = dinov2(images_norm)
            embeddings  = F.normalize(embeddings, dim=-1)
    
            # skip any NaN embeddings
            nan_mask = torch.isnan(embeddings).any(dim=1)
            if nan_mask.any():
                print(f'Skipping {nan_mask.sum()} NaN embeddings')
                embeddings = embeddings[~nan_mask]
                labels     = labels[~nan_mask]
    
            for c in range(10):
                mask = (labels == c)
                if mask.any():
                    prototype_sums[c]   += embeddings[mask].sum(dim=0)
                    prototype_counts[c] += mask.sum()

        prototypes = prototype_sums / prototype_counts.unsqueeze(1)
        prototypes = F.normalize(prototypes, dim=-1)
        
        # verify before saving
        print(f'Prototypes NaN: {torch.isnan(prototypes).any()}')
        print(f'Counts per class: {prototype_counts}')
        
        torch.save(prototypes, './dinov2_prototypes.pt')

In [ ]:
class DINOv2Classifier(torch.nn.Module):
    def __init__(self, dinov2, prototypes, normalize):
        super().__init__()
        self.dinov2 = dinov2
        self.prototypes = prototypes
        self.normalize = normalize


    def forward(self, x):
        x = self.normalize(x)
        z = self.dinov2(x)
        z = F.normalize(z, dim=-1)

        logits = z @ self.prototypes.T * 100.0
        return logits

In [ ]:
model = DINOv2Classifier(dinov2, prototypes, normalize).to(device).eval()

In [ ]:
print('\nMeasuring clean accuracy...')
correct = 0
total   = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        preds  = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

clean_acc = correct / total * 100
print(f'Clean accuracy: {clean_acc:.2f}%')

In [26]:
# use 200 samples instead of 10
x_sample = torch.stack([test_data[i][0] for i in range(200)]).to(device)
y_sample  = torch.tensor([test_data[i][1] for i in range(200)]).to(device)

# find which ones the model classifies correctly
with torch.no_grad():
    preds = []
    for i in range(0, 200, 10):
        logits = model(x_sample[i:i+10])
        preds.append(logits.argmax(dim=1))
    preds = torch.cat(preds)

correct_mask = (preds == y_sample)
print(f'Correctly classified: {correct_mask.sum()}/{len(y_sample)}')

# only attack correctly classified samples
x_correct = x_sample[correct_mask]
y_correct  = y_sample[correct_mask]

# run attack on correct samples only
adversary_pgd = AutoAttack(
    model, norm='Linf', eps=8/255,
    version='custom', attacks_to_run=['apgd-ce'],
    verbose=True
)
adversary_pgd.apgd.n_iter = 50

x_adv = adversary_pgd.run_standard_evaluation(
    x_correct, y_correct, bs=5
)

# measure embedding shift on correctly classified samples
with torch.no_grad():
    shifts = []
    for i in range(0, len(x_correct), 5):
        zc = F.normalize(dinov2(normalize(x_correct[i:i+5])), dim=-1)
        za = F.normalize(dinov2(normalize(x_adv[i:i+5])),    dim=-1)
        shifts.append((za - zc).norm(dim=-1))
    shifts = torch.cat(shifts)

print(f'Samples attacked:              {len(x_correct)}')
print(f'Mean embedding shift (DINOv2): {shifts.mean():.4f}')
print(f'Max  embedding shift (DINOv2): {shifts.max():.4f}')

Correctly classified: 20/200
using custom version including apgd-ce.
initial accuracy: 100.00%
apgd-ce - 1/4 - 0 out of 5 successfully perturbed
apgd-ce - 2/4 - 0 out of 5 successfully perturbed
apgd-ce - 3/4 - 0 out of 5 successfully perturbed
apgd-ce - 4/4 - 0 out of 5 successfully perturbed
robust accuracy after APGD-CE: 100.00% (total time 240.8 s)
max Linf perturbation: 0.00000, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 100.00%
Samples attacked:              20
Mean embedding shift (DINOv2): 0.0000
Max  embedding shift (DINOv2): 0.0000


In [31]:
print(f'Prototype dim: {prototypes.shape}')        # should match
print(f'DINOv2 output dim: {dinov2(normalize(x_sample[:1])).shape}')

Prototype dim: torch.Size([10, 1024])
DINOv2 output dim: torch.Size([1, 1024])


In [ ]:
print('\nRunning AutoAttack evaluation...')

# load first 512 test samples (following DiffPure evaluation protocol)
n_eval   = 512
x_test   = torch.stack([test_data[i][0] for i in range(n_eval)]).to(device)
y_test   = torch.tensor([test_data[i][1] for i in range(n_eval)]).to(device)

adversary = AutoAttack(
    model, norm='Linf', eps=8/255,
    version='standard',    # apgd-ce + apgd-dlr + fab + square
    verbose=True
)

x_adv_full, y_adv = adversary.run_standard_evaluation(
    x_test, y_test, bs=128, return_labels=True
)

robust_acc = (y_adv == y_test).float().mean().item() * 100
print(f'\n[FINAL] clean: {clean_acc:.2f}%  |  robust: {robust_acc:.2f}%')